In [1]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

### Download all data from Qdrant

In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [7]:
all_points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False   
)

In [8]:
all_points[0][0].payload

{'preprocessed_description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sitting

In [9]:
all_points[0]

[Record(id=0, payload={'preprocessed_description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the

In [10]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in all_points[0]]

In [11]:
all_context

[{'id': 'B0BRJS644Z',
  'text': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sitt

### Render a prompt to generate synthetic Eval reference dataset

In [12]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            }
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""

In [13]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, 

In [14]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text:
[
  {
    "id": "B0BRJS644Z",
    "text": "Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) \u3010Solve Your Cooling Problem\u3011\u00a0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. \u3010Speed\u00a0Control\u00a0Switch\u3011 The speed controller located on the cord allows you to adjust the fan\u2019s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. \u3010High Compatibility USB-Powered\u3011 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving 

In [15]:
response = openai.chat.completions.create(
    model="gpt-5.4",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

[
  {
    "reasoning": "This product clearly states it is a 120mm USB-powered cooling fan for routers, modems, receivers, DVRs, Xbox, TV boxes, and similar electronics, and it mentions the speed controller and dust filters.",
    "question": "Do you have a USB fan I can use to keep my router or TV box from overheating?",
    "chunk_ids": ["B0BRJS644Z"],
    "answer_example": "Yes. The Marame 120mm USB powered fan is designed to cool devices like routers, modems, receivers, DVRs, Xbox consoles, and TV boxes. It has a speed controller with off, low, medium, and high settings, and it also includes rubber feet and dust filters."
  },
  {
    "reasoning": "This product description specifically covers kids wireless headphones, including Bluetooth 5.0, a 3.5mm jack, foldable design, built-in mic, and up to 7 hours of battery life.",
    "question": "Do you have kids headphones that work wirelessly and also with a cable?",
    "chunk_ids": ["B09KQP2H7N"],
    "answer_example": "Yes. These kids

In [16]:
response.usage

CompletionUsage(completion_tokens=4797, prompt_tokens=19785, total_tokens=24582, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

In [17]:
json_output = response.choices[0].message.content

In [18]:
json_output = json_output.replace("```json", "").replace("```", "")

In [19]:
print(json_output)

[
  {
    "reasoning": "This product clearly states it is a 120mm USB-powered cooling fan for routers, modems, receivers, DVRs, Xbox, TV boxes, and similar electronics, and it mentions the speed controller and dust filters.",
    "question": "Do you have a USB fan I can use to keep my router or TV box from overheating?",
    "chunk_ids": ["B0BRJS644Z"],
    "answer_example": "Yes. The Marame 120mm USB powered fan is designed to cool devices like routers, modems, receivers, DVRs, Xbox consoles, and TV boxes. It has a speed controller with off, low, medium, and high settings, and it also includes rubber feet and dust filters."
  },
  {
    "reasoning": "This product description specifically covers kids wireless headphones, including Bluetooth 5.0, a 3.5mm jack, foldable design, built-in mic, and up to 7 hours of battery life.",
    "question": "Do you have kids headphones that work wirelessly and also with a cable?",
    "chunk_ids": ["B09KQP2H7N"],
    "answer_example": "Yes. These kids

In [20]:
json_output = json.loads(json_output)

In [21]:
json_output

[{'reasoning': 'This product clearly states it is a 120mm USB-powered cooling fan for routers, modems, receivers, DVRs, Xbox, TV boxes, and similar electronics, and it mentions the speed controller and dust filters.',
  'question': 'Do you have a USB fan I can use to keep my router or TV box from overheating?',
  'chunk_ids': ['B0BRJS644Z'],
  'answer_example': 'Yes. The Marame 120mm USB powered fan is designed to cool devices like routers, modems, receivers, DVRs, Xbox consoles, and TV boxes. It has a speed controller with off, low, medium, and high settings, and it also includes rubber feet and dust filters.'},
 {'reasoning': 'This product description specifically covers kids wireless headphones, including Bluetooth 5.0, a 3.5mm jack, foldable design, built-in mic, and up to 7 hours of battery life.',
  'question': 'Do you have kids headphones that work wirelessly and also with a cable?',
  'chunk_ids': ['B09KQP2H7N'],
  'answer_example': 'Yes. These kids wireless headphones support 

In [22]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0

for item in json_output:
    if len(item["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(item["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        impossible_counter += 1

In [23]:
print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Impossible questions: {impossible_counter}")

Single chunk questions: 15
Multiple chunk questions: 11
Impossible questions: 5


In [24]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B099N9F3FP")
            )
        ]
    )
)[0]

In [25]:
points[0].payload

{'preprocessed_description': 'AICHESON Laptop Cooler Pad with 6 Cooling Fans for 11" - 15.6" Laptops, RED Lights Super Laptop Cooling Fans - AICHESON cooling pads laptop is specially designed with 6 (70mm) cooling fans, each fan speed is 1500-2200RPM from level 1 to level 6, providing you adjustable airflows to lengthen the life of your laptop Ergonomic Heights Stands - Once the hook is dropped, it is locked in position, and it is very convenient to adjust the angles and heights, angles can change from 10 to 27 degrees, raising your laptop up 2.83" to 5.19", providing you a better and a more comfortable posture when using the laptop Low Noise and Easy Operation - Even structured with 6 cooling fans, the noise level is 25dBA - 28 dBA, which is just like someone whispers around your ear. And the LCD display control panel allows you to adjust fan speed easily. There is no need to wrap your hands behind the cooler to operate, which is very troublesome 3 Modes and Extra USB Ports - Long pre

In [26]:
def get_description(parent_asin: str) -> str:
    
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]

    return points[0].payload["preprocessed_description"]

In [27]:
get_description("B099N9F3FP")

'AICHESON Laptop Cooler Pad with 6 Cooling Fans for 11" - 15.6" Laptops, RED Lights Super Laptop Cooling Fans - AICHESON cooling pads laptop is specially designed with 6 (70mm) cooling fans, each fan speed is 1500-2200RPM from level 1 to level 6, providing you adjustable airflows to lengthen the life of your laptop Ergonomic Heights Stands - Once the hook is dropped, it is locked in position, and it is very convenient to adjust the angles and heights, angles can change from 10 to 27 degrees, raising your laptop up 2.83" to 5.19", providing you a better and a more comfortable posture when using the laptop Low Noise and Easy Operation - Even structured with 6 cooling fans, the noise level is 25dBA - 28 dBA, which is just like someone whispers around your ear. And the LCD display control panel allows you to adjust fan speed easily. There is no need to wrap your hands behind the cooler to operate, which is very troublesome 3 Modes and Extra USB Ports - Long pressing the ON/OFF button to ad

### Create Eval dataset in LangSmith

In [28]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [29]:
ls_client = Client()

In [30]:
dataset_name = "rag-evaluation-dataset"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset"
)

In [31]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"]
        },
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )